In [1]:
import os
import numpy as np
import cv2
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from pathlib import Path
import re
import imageio
from scipy import ndimage
from typing import List, Tuple, Optional

def natural_sort_key(text):
    """Natural sorting key for filenames with numbers (e.g., epoch_1, epoch_2, ..., epoch_10)"""
    return [int(c) if c.isdigit() else c.lower() for c in re.split('([0-9]+)', text)]

def load_images_from_folder(folder_path: str, extensions: List[str] = ['.png', '.jpg', '.jpeg']) -> Tuple[List[np.ndarray], List[str]]:
    """Load images from folder in natural order"""
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")
    
    # Get all image files
    image_files = []
    for ext in extensions:
        image_files.extend(folder.glob(f"*{ext}"))
        image_files.extend(folder.glob(f"*{ext.upper()}"))
    
    # Sort naturally (handles numbers correctly)
    image_files = sorted(image_files, key=lambda x: natural_sort_key(x.name))
    
    if not image_files:
        raise ValueError(f"No images found in {folder_path}")
    
    print(f"Found {len(image_files)} images in {folder_path}")
    
    # Load images
    images = []
    filenames = []
    for img_path in image_files:
        img = cv2.imread(str(img_path))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img_rgb)
            filenames.append(img_path.name)
        else:
            print(f"Warning: Could not load {img_path}")
    
    print(f"Successfully loaded {len(images)} images")
    return images, filenames

def interpolate_frames(img1: np.ndarray, img2: np.ndarray, num_interpolations: int = 3) -> List[np.ndarray]:
    """
    Create smooth interpolated frames between two images using multiple techniques
    
    Args:
        img1: First image (numpy array)
        img2: Second image (numpy array) 
        num_interpolations: Number of interpolated frames to create between img1 and img2
        
    Returns:
        List of interpolated frames (including original img1, but not img2)
    """
    if img1.shape != img2.shape:
        # Resize img2 to match img1 if shapes differ
        img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
    
    interpolated = [img1]  # Start with first image
    
    # Create interpolation weights
    for i in range(1, num_interpolations + 1):
        alpha = i / (num_interpolations + 1)  # Weight for img2
        beta = 1 - alpha  # Weight for img1
        
        # Method 1: Simple linear blending
        blended = cv2.addWeighted(img1.astype(np.float32), beta, 
                                 img2.astype(np.float32), alpha, 0)
        
        # Method 2: Add slight gaussian blur for smoother transitions
        kernel_size = max(1, int(alpha * 3))  # Gradually increase blur
        if kernel_size > 1 and kernel_size % 2 == 0:
            kernel_size += 1  # Ensure odd kernel size
            
        if kernel_size > 1:
            blended = cv2.GaussianBlur(blended, (kernel_size, kernel_size), 0.5)
        
        # Ensure values are in valid range
        blended = np.clip(blended, 0, 255).astype(np.uint8)
        interpolated.append(blended)
    
    return interpolated

def create_smooth_gif(folder_path: str, 
                     output_path: str = "smooth_animation.gif",
                     interpolations_per_frame: int = 2,
                     frame_duration: float = 0.3,
                     loop_back: bool = True,
                     resize_to: Optional[Tuple[int, int]] = None,
                     add_text_overlay: bool = True) -> str:
    """
    Create a smooth GIF with interpolated frames from images in a folder
    
    Args:
        folder_path: Path to folder containing images
        output_path: Output GIF file path
        interpolations_per_frame: Number of interpolated frames between each pair
        frame_duration: Duration of each frame in seconds
        loop_back: If True, creates a seamless loop by reversing back to start
        resize_to: Optional tuple (width, height) to resize images
        add_text_overlay: Add frame number overlay
        
    Returns:
        Path to created GIF file
    """
    
    # Load images
    images, filenames = load_images_from_folder(folder_path)
    
    if len(images) < 2:
        raise ValueError("Need at least 2 images to create interpolated GIF")
    
    print(f"Creating smooth GIF with {interpolations_per_frame} interpolations per frame...")
    
    # Resize images if requested
    if resize_to:
        print(f"Resizing images to {resize_to}")
        resized_images = []
        for img in images:
            resized = cv2.resize(img, resize_to, interpolation=cv2.INTER_LANCZOS4)
            resized_images.append(resized)
        images = resized_images
    
    # Create interpolated frames
    all_frames = []
    frame_labels = []
    
    for i in range(len(images)):
        current_img = images[i]
        next_img = images[(i + 1) % len(images)]  # Wrap around for seamless loop
        
        # Add text overlay if requested
        if add_text_overlay:
            img_with_text = current_img.copy()
            # Extract frame number from filename if possible
            frame_num = re.search(r'(\d+)', filenames[i])
            if frame_num:
                label = f"Frame {frame_num.group(1)}"
            else:
                label = f"Frame {i+1}"
            
            cv2.putText(img_with_text, label, (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            cv2.putText(img_with_text, label, (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 1)
            current_img = img_with_text
            frame_labels.append(label)
        
        # Get interpolated frames (don't include the next image to avoid duplication)
        if i < len(images) - 1 or loop_back:  # Interpolate unless it's the last frame and no loop
            interpolated = interpolate_frames(current_img, next_img, interpolations_per_frame)
            all_frames.extend(interpolated)
        else:
            all_frames.append(current_img)
    
    # If loop_back is True, add reverse sequence (excluding first and last to avoid duplication)
    if loop_back and len(images) > 2:
        print("Adding reverse sequence for seamless loop...")
        reverse_frames = all_frames[-(interpolations_per_frame+1):-1]  # Exclude the last frame
        reverse_frames.reverse()
        all_frames.extend(reverse_frames)
    
    print(f"Total frames in animation: {len(all_frames)}")
    
    # Convert to PIL Images for GIF creation
    pil_frames = []
    for frame in all_frames:
        pil_img = Image.fromarray(frame)
        pil_frames.append(pil_img)
    
    # Create GIF
    print(f"Saving GIF to {output_path}...")
    duration_ms = int(frame_duration * 1000)  # Convert to milliseconds
    
    pil_frames[0].save(
        output_path,
        save_all=True,
        append_images=pil_frames[1:],
        duration=duration_ms,
        loop=0,  # Infinite loop
        optimize=True
    )
    
    # Get file size
    file_size = os.path.getsize(output_path)
    print(f"✅ GIF created successfully!")
    print(f"   📁 File: {output_path}")
    print(f"   📊 Size: {file_size / (1024*1024):.1f} MB")
    print(f"   🎬 Frames: {len(all_frames)}")
    print(f"   ⏱️  Duration per frame: {frame_duration}s")
    print(f"   🔄 Total duration: ~{len(all_frames) * frame_duration:.1f}s")
    
    return output_path

# Example usage function
def demo_smooth_gif():
    """Demo function showing how to use the GIF creation tools"""
    
    # Example 1: Create GIF from epoch overlays
    folder_path = "epoch_overlays"  # Your epoch overlay folder
    
    if os.path.exists(folder_path):
        print("Creating smooth GIF from epoch overlays...")
        
        # Create a smooth training progress GIF
        gif_path = create_smooth_gif(
            folder_path=folder_path,
            output_path="training_progress_smooth.gif",
            interpolations_per_frame=3,  # 3 interpolated frames between each epoch
            frame_duration=0.4,          # 400ms per frame
            loop_back=False,             # Don't reverse (training is linear progression)
            resize_to=(800, 600),        # Resize for smaller file size
            add_text_overlay=True        # Show epoch numbers
        )
        
        return gif_path
    else:
        print(f"Folder '{folder_path}' not found. Please specify the correct path to your images.")
        return None

print("🎬 Smooth GIF Creator Functions Loaded!")
print("\nUsage:")
print("1. demo_smooth_gif() - Create GIF from epoch_overlays folder")
print("2. create_smooth_gif(folder_path, output_path, ...) - Custom GIF creation")
print("3. load_images_from_folder(folder_path) - Load and sort images")
print("\nExample:")
print('create_smooth_gif("your_folder", "output.gif", interpolations_per_frame=4, frame_duration=0.3)')

🎬 Smooth GIF Creator Functions Loaded!

Usage:
1. demo_smooth_gif() - Create GIF from epoch_overlays folder
2. create_smooth_gif(folder_path, output_path, ...) - Custom GIF creation
3. load_images_from_folder(folder_path) - Load and sort images

Example:
create_smooth_gif("your_folder", "output.gif", interpolations_per_frame=4, frame_duration=0.3)


In [2]:
# Quick function to create GIF from your epoch overlays
def create_training_progress_gif(
    input_folder: str = "epoch_overlays",
    output_file: str = "training_progress.gif",
    interpolation_frames: int = 2,
    speed: str = "normal"  # "slow", "normal", "fast"
):
    """
    One-click function to create a smooth training progress GIF
    
    Args:
        input_folder: Folder containing epoch overlay images
        output_file: Output GIF filename
        interpolation_frames: Number of frames to interpolate between each epoch (1-5 recommended)
        speed: Animation speed - "slow" (0.5s), "normal" (0.3s), "fast" (0.15s)
    """
    
    # Speed settings
    speed_map = {
        "slow": 0.5,
        "normal": 0.3, 
        "fast": 0.15
    }
    frame_duration = speed_map.get(speed, 0.3)
    
    try:
        # Check if folder exists
        if not os.path.exists(input_folder):
            print(f"❌ Folder '{input_folder}' not found!")
            print("Available folders:", [d for d in os.listdir('.') if os.path.isdir(d)])
            return None
            
        # Create the GIF
        gif_path = create_smooth_gif(
            folder_path=input_folder,
            output_path=output_file,
            interpolations_per_frame=interpolation_frames,
            frame_duration=frame_duration,
            loop_back=False,  # Training is linear progression
            resize_to=(1000, 750),  # Good size for viewing
            add_text_overlay=True   # Show epoch numbers
        )
        
        print(f"\n🎉 Success! Your training progress GIF is ready: {gif_path}")
        return gif_path
        
    except Exception as e:
        print(f"❌ Error creating GIF: {str(e)}")
        return None

# Run this to create your GIF right now!
print("🚀 Ready to create your training progress GIF!")
print("\nRun one of these commands:")
print("📹 create_training_progress_gif()  # Default settings")
print("📹 create_training_progress_gif(interpolation_frames=4, speed='slow')  # More frames, slower")
print("📹 create_training_progress_gif(output_file='my_training.gif', speed='fast')  # Custom name, faster")

🚀 Ready to create your training progress GIF!

Run one of these commands:
📹 create_training_progress_gif()  # Default settings
📹 create_training_progress_gif(interpolation_frames=4, speed='slow')  # More frames, slower
📹 create_training_progress_gif(output_file='my_training.gif', speed='fast')  # Custom name, faster


In [4]:
# Execute this cell to create your training progress GIF!

# Check what's in the current directory
print("Current directory contents:")
for item in sorted(os.listdir('.')):
    if os.path.isdir(item):
        print(f"📁 {item}/")
    elif item.endswith(('.png', '.jpg', '.jpeg', '.gif')):
        print(f"🖼️  {item}")

print("\n" + "="*50)

# Create the GIF from your epoch overlays
result = create_training_progress_gif(
    input_folder="epoch_overlays",           # Your epoch overlay folder  
    output_file="training_evolution.gif",    # Output filename
    interpolation_frames=3,                  # 3 smooth frames between each epoch
    speed="fast"                          # Normal speed (0.3s per frame)
)

if result:
    print(f"\n✨ Your smooth training progress GIF is ready!")
    print(f"📂 Location: {os.path.abspath(result)}")
    
    # Optional: Display file info
    if os.path.exists(result):
        size_mb = os.path.getsize(result) / (1024 * 1024)
        print(f"📏 File size: {size_mb:.1f} MB")
        
        # You can view it in VS Code by right-clicking the file in explorer
        print(f"\n💡 To view: Right-click '{result}' in VS Code Explorer → 'Open Preview'")
else:
    print("\n❌ GIF creation failed. Check the error messages above.")

Current directory contents:
📁 .git/
📁 .github/
🖼️  Dataset Sample Visualization.png
📁 data/
📁 epoch_overlays/
📁 examples/
📁 logs/
📁 models/
🖼️  overlay_visualization.png
🖼️  prediction_comparison.png
📁 test_data/
📁 tests/
🖼️  training_evolution.gif
🖼️  training_history.png
📁 unet_roi/
📁 unet_ultrasound_roi.egg-info/

Found 50 images in epoch_overlays
Successfully loaded 50 images
Creating smooth GIF with 3 interpolations per frame...
Resizing images to (1000, 750)
Successfully loaded 50 images
Creating smooth GIF with 3 interpolations per frame...
Resizing images to (1000, 750)
Total frames in animation: 197
Saving GIF to training_evolution.gif...
Total frames in animation: 197
Saving GIF to training_evolution.gif...
✅ GIF created successfully!
   📁 File: training_evolution.gif
   📊 Size: 24.5 MB
   🎬 Frames: 197
   ⏱️  Duration per frame: 0.15s
   🔄 Total duration: ~29.5s

🎉 Success! Your training progress GIF is ready: training_evolution.gif

✨ Your smooth training progress GIF is re